<a href="https://colab.research.google.com/github/Ayseatci/DI725_Assignment1/blob/dev/notebooks/Base_RoBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base RoBERTa Model

This notebook consists of the initial experiments of the base RoBERTa model that only utilizes text/conversation in the dataset for sentiment prediction.

In [9]:
!git clone -b dev https://github.com/Ayseatci/DI725_Assignment1.git

Cloning into 'DI725_Assignment1'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 85 (delta 38), reused 57 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 852.17 KiB | 18.94 MiB/s, done.
Resolving deltas: 100% (38/38), done.


In [10]:
%cd DI725_Assignment1

/content/DI725_Assignment1/DI725_Assignment1


Head truncation mode base RoBERTa experiment and results

In [11]:
!pip install wandb
!wandb login
import wandb

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [12]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import random


SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to: {seed}")

set_seed(SEED)

# Configuration
TRUNCATION_MODE = 'head'
print(f"--- Running experiment with TRUNCATION_MODE: {TRUNCATION_MODE} ---")

# Initialize W&B
wandb.init(
    project="DI725_Assignment_1",
    name=f"base-roberta-{TRUNCATION_MODE}-seed{SEED}",
    config={
        "model": "roberta-base",
        "truncation": TRUNCATION_MODE,
        "epochs": 3,
        "lr": 2e-5,
        "seed": SEED
    }
)


# Load preprocessed data
data = pd.read_csv("data/preprocessed/preprocessed_train.csv")
data = data.dropna(subset=['text', 'label'])

# Label mapping adjusted according to preprocessing steps
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

# Train and Validation split
train_df, val_df = train_test_split(
    data[['text', 'label']],
    test_size=0.2,
    stratify=data['label'],
    random_state=SEED  # global seed
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Tokenizer and special tokens
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

# Speaker tags for customer and agent
new_tokens = ["<cust>", "<agent>"]
tokenizer.add_tokens(new_tokens)

# Tokenization
def tokenize_experiment(examples):
    # Disable automatic special tokens
    outputs = tokenizer(examples["text"], add_special_tokens=False)

    input_ids = []
    attention_masks = []

    for ids in outputs["input_ids"]:
        # Roberta max length is 512, 2 slots reserved for <s> (BOS) and </s> (EOS)
        if len(ids) > 510:
            if TRUNCATION_MODE == 'head':
                # Head, keep the start of the chat
                processed = ids[:510]
            elif TRUNCATION_MODE == 'tail':
                # Tail, keep the end of the chat
                processed = ids[-510:]
            else:
                # Both head and tail,255 from start, 255 from end
                processed = ids[:255] + ids[-255:]

            combined = [tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id]
        else:
            combined = [tokenizer.bos_token_id] + ids + [tokenizer.eos_token_id]

        input_ids.append(combined)
        attention_masks.append([1] * len(combined))

    return {"input_ids": input_ids, "attention_mask": attention_masks}

train_dataset = train_dataset.map(tokenize_experiment, batched=True)
val_dataset = val_dataset.map(tokenize_experiment, batched=True)

# Model Initialization
model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

# Resizing embeddings due to added special tokens <cust> and <agent>
model.resize_token_embeddings(len(tokenizer))

# Training setup
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# Training arguments
training_args = TrainingArguments(
    output_dir=f"./roberta_{TRUNCATION_MODE}",
    seed=SEED,
    data_seed=SEED,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="wandb"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# Experiment execution
trainer.train()

# Final Evaluation
print(f"\nFinal Classification Report ({TRUNCATION_MODE}):")
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(preds.label_ids, y_pred, target_names=list(id2label.values())))

wandb.finish()

--- Running experiment with TRUNCATION_MODE: head ---


Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Classification Report (head):


              precision    recall  f1-score   support

    negative       0.89      0.88      0.88        82
     neutral       0.88      0.92      0.90       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.89       194
   macro avg       0.59      0.60      0.59       194
weighted avg       0.87      0.89      0.88       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁▃█
eval/f1_macro,▁▃█
eval/loss,▃█▁
eval/runtime,▄█▁
eval/samples_per_second,▅▁█
eval/steps_per_second,▅▁█
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Tail truncation mode base RoBERTa experiment and results

In [13]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import random


SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to: {seed}")

set_seed(SEED)

# Configuration
TRUNCATION_MODE = 'tail'
print(f"--- Running experiment with TRUNCATION_MODE: {TRUNCATION_MODE} ---")

# Initialize W&B
wandb.init(
    project="DI725_Assignment_1",
    name=f"base-roberta-{TRUNCATION_MODE}-seed{SEED}",
    config={
        "model": "roberta-base",
        "truncation": TRUNCATION_MODE,
        "epochs": 3,
        "lr": 2e-5,
        "seed": SEED
    }
)


# Load preprocessed data
data = pd.read_csv("data/preprocessed/preprocessed_train.csv")
data = data.dropna(subset=['text', 'label'])

# Label mapping adjusted according to preprocessing steps
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

# Train and Validation split
train_df, val_df = train_test_split(
    data[['text', 'label']],
    test_size=0.2,
    stratify=data['label'],
    random_state=SEED  # global seed
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Tokenizer and special tokens
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

# Speaker tags for customer and agent
new_tokens = ["<cust>", "<agent>"]
tokenizer.add_tokens(new_tokens)

# Tokenization
def tokenize_experiment(examples):
    # Disable automatic special tokens
    outputs = tokenizer(examples["text"], add_special_tokens=False)

    input_ids = []
    attention_masks = []

    for ids in outputs["input_ids"]:
        # Roberta max length is 512, 2 slots reserved for <s> (BOS) and </s> (EOS)
        if len(ids) > 510:
            if TRUNCATION_MODE == 'head':
                # Head, keep the start of the chat
                processed = ids[:510]
            elif TRUNCATION_MODE == 'tail':
                # Tail, keep the end of the chat
                processed = ids[-510:]
            else:
                # Both head and tail,255 from start, 255 from end
                processed = ids[:255] + ids[-255:]

            combined = [tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id]
        else:
            combined = [tokenizer.bos_token_id] + ids + [tokenizer.eos_token_id]

        input_ids.append(combined)
        attention_masks.append([1] * len(combined))

    return {"input_ids": input_ids, "attention_mask": attention_masks}

train_dataset = train_dataset.map(tokenize_experiment, batched=True)
val_dataset = val_dataset.map(tokenize_experiment, batched=True)

# Model Initialization
model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

# Resizing embeddings due to added special tokens <cust> and <agent>
model.resize_token_embeddings(len(tokenizer))

# Training setup
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# Training arguments
training_args = TrainingArguments(
    output_dir=f"./roberta_{TRUNCATION_MODE}",
    seed=SEED,
    data_seed=SEED,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="wandb"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# Experiment execution
trainer.train()

# Final Evaluation
print(f"\nFinal Classification Report ({TRUNCATION_MODE}):")
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(preds.label_ids, y_pred, target_names=list(id2label.values())))

wandb.finish()

Global seed set to: 42
--- Running experiment with TRUNCATION_MODE: tail ---


Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Classification Report (tail):


              precision    recall  f1-score   support

    negative       0.91      0.89      0.90        82
     neutral       0.89      0.94      0.91       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.90       194
   macro avg       0.60      0.61      0.61       194
weighted avg       0.89      0.90      0.89       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁▂█
eval/f1_macro,▁▂█
eval/loss,▂█▁
eval/runtime,▂▁█
eval/samples_per_second,▇█▁
eval/steps_per_second,▇█▁
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


Mixed truncation mode base RoBERTa experiment and results

In [14]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import random


SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to: {seed}")

set_seed(SEED)

# Configuration
TRUNCATION_MODE = 'mixed'
print(f"--- Running experiment with TRUNCATION_MODE: {TRUNCATION_MODE} ---")

# Initialize W&B
wandb.init(
    project="DI725_Assignment_1",
    name=f"base-roberta-{TRUNCATION_MODE}-seed{SEED}",
    config={
        "model": "roberta-base",
        "truncation": TRUNCATION_MODE,
        "epochs": 3,
        "lr": 2e-5,
        "seed": SEED
    }
)


# Load preprocessed data
data = pd.read_csv("data/preprocessed/preprocessed_train.csv")
data = data.dropna(subset=['text', 'label'])

# Label mapping adjusted according to preprocessing steps
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

# Train and Validation split
train_df, val_df = train_test_split(
    data[['text', 'label']],
    test_size=0.2,
    stratify=data['label'],
    random_state=SEED  # global seed
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Tokenizer and special tokens
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

# Speaker tags for customer and agent
new_tokens = ["<cust>", "<agent>"]
tokenizer.add_tokens(new_tokens)

# Tokenization
def tokenize_experiment(examples):
    # Disable automatic special tokens
    outputs = tokenizer(examples["text"], add_special_tokens=False)

    input_ids = []
    attention_masks = []

    for ids in outputs["input_ids"]:
        # Roberta max length is 512, 2 slots reserved for <s> (BOS) and </s> (EOS)
        if len(ids) > 510:
            if TRUNCATION_MODE == 'head':
                # Head, keep the start of the chat
                processed = ids[:510]
            elif TRUNCATION_MODE == 'tail':
                # Tail, keep the end of the chat
                processed = ids[-510:]
            else:
                # Both head and tail,255 from start, 255 from end
                processed = ids[:255] + ids[-255:]

            combined = [tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id]
        else:
            combined = [tokenizer.bos_token_id] + ids + [tokenizer.eos_token_id]

        input_ids.append(combined)
        attention_masks.append([1] * len(combined))

    return {"input_ids": input_ids, "attention_mask": attention_masks}

train_dataset = train_dataset.map(tokenize_experiment, batched=True)
val_dataset = val_dataset.map(tokenize_experiment, batched=True)

# Model Initialization
model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

# Resizing embeddings due to added special tokens <cust> and <agent>
model.resize_token_embeddings(len(tokenizer))

# Training setup
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# Training arguments
training_args = TrainingArguments(
    output_dir=f"./roberta_{TRUNCATION_MODE}",
    seed=SEED,
    data_seed=SEED,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="wandb"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# Experiment execution
trainer.train()

# Final Evaluation
print(f"\nFinal Classification Report ({TRUNCATION_MODE}):")
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(preds.label_ids, y_pred, target_names=list(id2label.values())))

wandb.finish()

Global seed set to: 42
--- Running experiment with TRUNCATION_MODE: mixed ---


Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Classification Report (mixed):


              precision    recall  f1-score   support

    negative       0.92      0.89      0.91        82
     neutral       0.90      0.94      0.92       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.91       194
   macro avg       0.61      0.61      0.61       194
weighted avg       0.89      0.91      0.90       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/accuracy,▁██
eval/f1_macro,▁██
eval/loss,█▁▂
eval/runtime,█▁▆
eval/samples_per_second,▁█▂
eval/steps_per_second,▁█▂
test/accuracy,▁
test/f1_macro,▁
test/loss,▁
test/runtime,▁
+7,...


The experiment results of the based RoBERTa model with different truncation modes show that the textual context alone is insufficient for sentiment classification in this dataset.

Neutral and positive texts appear to have high lexical overlap therefore a pure RoBERTa model cannot distinguish between these two classes which can be seen by the consistent 0.00 F1-score and recall for the positive class across different truncation strategies.

While head truncation provided slightly higher precision for negative cases, and tail truncation improved overall macro F1, the model consistently fails to capture the signals of positive sentiment.

As a result, a pure transformer based approach is inadequate for the imbalanced dataset. To solve this problem, additional features such as the numerical and categorical features should be integrated into a hybrid architecture so that the model can support the context.

## Additional experiments with tail truncation configuration

Experiment with focal loss

In [17]:
import random
import os
import torch
import numpy as np
import pandas as pd
from torch import nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import wandb

SEED = 42

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Experiment Settings are 'weighted', 'oversample', 'downsample', or 'focal'
EXPERIMENT_TYPE = 'focal'
PROJECT_NAME = "DI725_Assignment_1"
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-seed{SEED}"

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

# Data loading
data = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
id2label = {0: "negative", 1: "neutral", 2: "positive"}

train_df, val_df = train_test_split(
    data[['text', 'label']],
    test_size=0.2,
    stratify=data['label'],
    random_state=SEED # Seeded Split
)

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    max_size = counts.max()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == max_size:
            # Keep majority class as is
            lst.append(train_df[train_df['label'] == i])
        else:
            # 4 times the current size, but don't exceed the majority class size
            target_size = min(n_current * 4, max_size)

            if n_current > 0:
                upsampled = train_df[train_df['label'] == i].sample(
                    target_size,
                    replace=True,
                    random_state=SEED
                )
                lst.append(upsampled)

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Custom Oversampled Training Size: {len(train_df)}")
    print("New Class Distribution:\n", train_df['label'].value_counts())

elif EXPERIMENT_TYPE == 'downsample':
    min_size = train_df['label'].value_counts().min()
    lst = [train_df[train_df['label'] == i].sample(min_size, random_state=SEED) for i in range(3)]
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ImbalanceTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculating class weights
        weights = compute_class_weight('balanced', classes=np.unique(data['label']), y=data['label'])
        class_weights = torch.tensor(weights, dtype=torch.float).to(model.device)

        if EXPERIMENT_TYPE == 'weighted':
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        elif EXPERIMENT_TYPE == 'focal':
            loss_fct = FocalLoss(alpha=class_weights, gamma=2.0)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Training execution
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.add_tokens(["<cust>", "<agent>"])

def tokenize_tail(examples):
    outputs = tokenizer(examples["text"], add_special_tokens=False)
    input_ids = []
    for ids in outputs["input_ids"]:
        processed = ids[-510:] if len(ids) > 510 else ids
        input_ids.append([tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id])
    return {"input_ids": input_ids}

train_dataset = train_dataset.map(tokenize_tail, batched=True)
val_dataset = val_dataset.map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir=f"./results_{EXPERIMENT_TYPE}",
    eval_strategy="epoch",
    logging_steps=10,
    report_to="wandb",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available()
)

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")

preds_output = trainer.predict(val_dataset)
logits = preds_output.predictions
y_true = preds_output.label_ids
y_pred = np.argmax(logits, axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
)
print(report)

wandb.log({"final_classification_report": report})

wandb.finish()

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: focal (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.90      0.90      0.90        82
     neutral       0.90      0.93      0.91       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.90       194
   macro avg       0.60      0.61      0.61       194
weighted avg       0.89      0.90      0.89       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/loss,█▁▁
eval/runtime,▁▃█
eval/samples_per_second,█▆▁
eval/steps_per_second,█▆▁
test/loss,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇██████
+3,...


Experiment with oversample

In [18]:
import random
import os
import torch
import numpy as np
import pandas as pd
from torch import nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import wandb

SEED = 42

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Experiment Settings are 'weighted', 'oversample', 'downsample', or 'focal'
EXPERIMENT_TYPE = 'weighted'
PROJECT_NAME = "DI725_Assignment_1"
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-seed{SEED}"

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

# Data loading
data = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
id2label = {0: "negative", 1: "neutral", 2: "positive"}

train_df, val_df = train_test_split(
    data[['text', 'label']],
    test_size=0.2,
    stratify=data['label'],
    random_state=SEED # Seeded Split
)

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    max_size = counts.max()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == max_size:
            # Keep majority class as is
            lst.append(train_df[train_df['label'] == i])
        else:
            # 4 times the current size, but don't exceed the majority class size
            target_size = min(n_current * 4, max_size)

            if n_current > 0:
                upsampled = train_df[train_df['label'] == i].sample(
                    target_size,
                    replace=True,
                    random_state=SEED
                )
                lst.append(upsampled)

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Custom Oversampled Training Size: {len(train_df)}")
    print("New Class Distribution:\n", train_df['label'].value_counts())

elif EXPERIMENT_TYPE == 'downsample':
    min_size = train_df['label'].value_counts().min()
    lst = [train_df[train_df['label'] == i].sample(min_size, random_state=SEED) for i in range(3)]
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ImbalanceTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculating class weights
        weights = compute_class_weight('balanced', classes=np.unique(data['label']), y=data['label'])
        class_weights = torch.tensor(weights, dtype=torch.float).to(model.device)

        if EXPERIMENT_TYPE == 'weighted':
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        elif EXPERIMENT_TYPE == 'focal':
            loss_fct = FocalLoss(alpha=class_weights, gamma=2.0)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Training execution
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.add_tokens(["<cust>", "<agent>"])

def tokenize_tail(examples):
    outputs = tokenizer(examples["text"], add_special_tokens=False)
    input_ids = []
    for ids in outputs["input_ids"]:
        processed = ids[-510:] if len(ids) > 510 else ids
        input_ids.append([tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id])
    return {"input_ids": input_ids}

train_dataset = train_dataset.map(tokenize_tail, batched=True)
val_dataset = val_dataset.map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir=f"./results_{EXPERIMENT_TYPE}",
    eval_strategy="epoch",
    logging_steps=10,
    report_to="wandb",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available()
)

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")

preds_output = trainer.predict(val_dataset)
logits = preds_output.predictions
y_true = preds_output.label_ids
y_pred = np.argmax(logits, axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
)
print(report)

wandb.log({"final_classification_report": report})

wandb.finish()

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: weighted (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.89      0.89      0.89        82
     neutral       0.89      0.92      0.90       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.89       194
   macro avg       0.59      0.60      0.60       194
weighted avg       0.88      0.89      0.88       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/loss,▁█▂
eval/runtime,▃▁█
eval/samples_per_second,▆█▁
eval/steps_per_second,▆█▁
test/loss,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇██████
+3,...


Additional experiment with oversampling

In [19]:
import random
import os
import torch
import numpy as np
import pandas as pd
from torch import nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import wandb

SEED = 42

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Experiment Settings are 'weighted', 'oversample', 'downsample', or 'focal'
EXPERIMENT_TYPE = 'oversample'
PROJECT_NAME = "DI725_Assignment_1"
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-seed{SEED}"

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

# Data loading
data = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
id2label = {0: "negative", 1: "neutral", 2: "positive"}

train_df, val_df = train_test_split(
    data[['text', 'label']],
    test_size=0.2,
    stratify=data['label'],
    random_state=SEED # Seeded Split
)

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    max_size = counts.max()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == max_size:
            # Keep majority class as is
            lst.append(train_df[train_df['label'] == i])
        else:
            # 4 times the current size, but don't exceed the majority class size
            target_size = min(n_current * 4, max_size)

            if n_current > 0:
                upsampled = train_df[train_df['label'] == i].sample(
                    target_size,
                    replace=True,
                    random_state=SEED
                )
                lst.append(upsampled)

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Custom Oversampled Training Size: {len(train_df)}")
    print("New Class Distribution:\n", train_df['label'].value_counts())

elif EXPERIMENT_TYPE == 'downsample':
    min_size = train_df['label'].value_counts().min()
    lst = [train_df[train_df['label'] == i].sample(min_size, random_state=SEED) for i in range(3)]
    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ImbalanceTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculating class weights
        weights = compute_class_weight('balanced', classes=np.unique(data['label']), y=data['label'])
        class_weights = torch.tensor(weights, dtype=torch.float).to(model.device)

        if EXPERIMENT_TYPE == 'weighted':
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        elif EXPERIMENT_TYPE == 'focal':
            loss_fct = FocalLoss(alpha=class_weights, gamma=2.0)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Training execution
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.add_tokens(["<cust>", "<agent>"])

def tokenize_tail(examples):
    outputs = tokenizer(examples["text"], add_special_tokens=False)
    input_ids = []
    for ids in outputs["input_ids"]:
        processed = ids[-510:] if len(ids) > 510 else ids
        input_ids.append([tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id])
    return {"input_ids": input_ids}

train_dataset = train_dataset.map(tokenize_tail, batched=True)
val_dataset = val_dataset.map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir=f"./results_{EXPERIMENT_TYPE}",
    eval_strategy="epoch",
    logging_steps=10,
    report_to="wandb",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available()
)

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")

preds_output = trainer.predict(val_dataset)
logits = preds_output.predictions
y_true = preds_output.label_ids
y_pred = np.argmax(logits, axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
)
print(report)

wandb.log({"final_classification_report": report})

wandb.finish()

Custom Oversampled Training Size: 922
New Class Distribution:
 label
0    433
1    433
2     56
Name: count, dtype: int64


Map:   0%|          | 0/922 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (538 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss
1,0.396519,0.416987
2,0.335870,0.595589
3,0.186768,0.363827


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: oversample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.94      0.82      0.88        82
     neutral       0.85      0.94      0.90       109
    positive       0.00      0.00      0.00         3

    accuracy                           0.88       194
   macro avg       0.60      0.59      0.59       194
weighted avg       0.88      0.88      0.87       194



eval/loss,▃█▁
eval/runtime,▂▁█
eval/samples_per_second,▇█▁
eval/steps_per_second,▇█▁
test/loss,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██████
+3,...


In [20]:
import os

output_path = "./results_oversample"
if os.path.exists(output_path):
    checkpoints = [d for d in os.listdir(output_path) if "checkpoint" in d]
    checkpoints.sort(key=lambda x: int(x.split('-')[1]))
    print("Found checkpoints:", checkpoints)
    print("Latest checkpoint:", os.path.join(output_path, checkpoints[-1]))
else:
    print("Directory not found. Check your 'output_dir' setting in TrainingArguments.")

Found checkpoints: ['checkpoint-348']
Latest checkpoint: ./results_oversample/checkpoint-348


Load model weights for oversampling and use on test data

In [21]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer

model_path = "./results_oversample/checkpoint-348"

model = RobertaForSequenceClassification.from_pretrained(model_path)
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.add_tokens(["<cust>", "<agent>"])

test_df = pd.read_csv("data/preprocessed/preprocessed_test.csv")
test_dataset = Dataset.from_pandas(test_df)

def tokenize_tail(examples):
    outputs = tokenizer(examples["text"], add_special_tokens=False)
    input_ids = []
    for ids in outputs["input_ids"]:
        processed = ids[-510:] if len(ids) > 510 else ids
        input_ids.append([tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id])
    return {"input_ids": input_ids}

test_dataset = test_dataset.map(tokenize_tail, batched=True)

tester = Trainer(model=model, data_collator=DataCollatorWithPadding(tokenizer=tokenizer))
raw_preds = tester.predict(test_dataset)

y_pred = np.argmax(raw_preds.predictions, axis=1)
y_true = test_df['label'].values

print("--- Test Set Results (Oversampled Model) ---")
print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (563 > 512). Running this sequence through the model will result in indexing errors


--- Test Set Results (Oversampled Model) ---
              precision    recall  f1-score   support

    negative       1.00      0.90      0.95        10
     neutral       0.59      1.00      0.74        10
    positive       1.00      0.40      0.57        10

    accuracy                           0.77        30
   macro avg       0.86      0.77      0.75        30
weighted avg       0.86      0.77      0.75        30



Experiment with downsample

In [22]:
import random
import os
import torch
import numpy as np
import pandas as pd
from torch import nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import wandb

SEED = 42

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Experiment Settings are 'weighted', 'oversample', 'downsample', or 'focal'
EXPERIMENT_TYPE = 'downsample'
PROJECT_NAME = "DI725_Assignment_1"
RUN_NAME = f"roberta-tail-{EXPERIMENT_TYPE}-seed{SEED}"

wandb.init(project=PROJECT_NAME, name=RUN_NAME, reinit=True)

# Data loading
data = pd.read_csv("data/preprocessed/preprocessed_train.csv").dropna(subset=['text', 'label'])
id2label = {0: "negative", 1: "neutral", 2: "positive"}

train_df, val_df = train_test_split(
    data[['text', 'label']],
    test_size=0.2,
    stratify=data['label'],
    random_state=SEED # Seeded Split
)

if EXPERIMENT_TYPE == 'oversample':
    counts = train_df['label'].value_counts()
    max_size = counts.max()

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current == max_size:
            # Keep majority class as is
            lst.append(train_df[train_df['label'] == i])
        else:
            # 4 times the current size, but don't exceed the majority class size
            target_size = min(n_current * 4, max_size)

            if n_current > 0:
                upsampled = train_df[train_df['label'] == i].sample(
                    target_size,
                    replace=True,
                    random_state=SEED
                )
                lst.append(upsampled)

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Custom Oversampled Training Size: {len(train_df)}")
    print("New Class Distribution:\n", train_df['label'].value_counts())

elif EXPERIMENT_TYPE == 'downsample':
    counts = train_df['label'].value_counts()
    min_size = counts.min()

    multiplier = 10
    target_cap = min_size * multiplier

    lst = []
    for i in range(3):
        n_current = counts.get(i, 0)

        if n_current <= target_cap:
            # If the class is already small, keep it all
            lst.append(train_df[train_df['label'] == i])
        else:
            # Downsample the majority class to the cap
            downsampled = train_df[train_df['label'] == i].sample(
                target_cap,
                random_state=SEED
            )
            lst.append(downsampled)

    train_df = pd.concat(lst).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Soft Downsampled Training Size: {len(train_df)}")
    print("New Class Distribution:\n", train_df['label'].value_counts())

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ImbalanceTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculating class weights
        weights = compute_class_weight('balanced', classes=np.unique(data['label']), y=data['label'])
        class_weights = torch.tensor(weights, dtype=torch.float).to(model.device)

        if EXPERIMENT_TYPE == 'weighted':
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        elif EXPERIMENT_TYPE == 'focal':
            loss_fct = FocalLoss(alpha=class_weights, gamma=2.0)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Training execution
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.add_tokens(["<cust>", "<agent>"])

def tokenize_tail(examples):
    outputs = tokenizer(examples["text"], add_special_tokens=False)
    input_ids = []
    for ids in outputs["input_ids"]:
        processed = ids[-510:] if len(ids) > 510 else ids
        input_ids.append([tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id])
    return {"input_ids": input_ids}

train_dataset = train_dataset.map(tokenize_tail, batched=True)
val_dataset = val_dataset.map(tokenize_tail, batched=True)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir=f"./results_{EXPERIMENT_TYPE}",
    eval_strategy="epoch",
    logging_steps=10,
    report_to="wandb",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available()
)

trainer = ImbalanceTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

print(f"\n--- Final Classification Report: {EXPERIMENT_TYPE} (Tail Truncation) ---")

preds_output = trainer.predict(val_dataset)
logits = preds_output.predictions
y_true = preds_output.label_ids
y_pred = np.argmax(logits, axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
)
print(report)

wandb.log({"final_classification_report": report})

wandb.finish()

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Classification Report: downsample (Tail Truncation) ---


              precision    recall  f1-score   support

    negative       0.52      0.90      0.66        82
     neutral       0.00      0.00      0.00       109
    positive       0.06      1.00      0.11         3

    accuracy                           0.40       194
   macro avg       0.19      0.63      0.26       194
weighted avg       0.22      0.40      0.28       194



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


eval/loss,█▅▁
eval/runtime,▇▁█
eval/samples_per_second,▂█▁
eval/steps_per_second,▂█▁
test/loss,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▃▅██
train/global_step,▁▃▅████
+3,...


In [23]:
import os

output_path = "./results_downsample"
if os.path.exists(output_path):
    checkpoints = [d for d in os.listdir(output_path) if "checkpoint" in d]
    checkpoints.sort(key=lambda x: int(x.split('-')[1]))
    print("Found checkpoints:", checkpoints)
    print("Latest checkpoint:", os.path.join(output_path, checkpoints[-1]))
else:
    print("Directory not found. Check your 'output_dir' setting in TrainingArguments.")

Found checkpoints: ['checkpoint-18']
Latest checkpoint: ./results_downsample/checkpoint-18


In [24]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer

model_path = "./results_downsample/checkpoint-18"

model = RobertaForSequenceClassification.from_pretrained(model_path)
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.add_tokens(["<cust>", "<agent>"])

test_df = pd.read_csv("data/preprocessed/preprocessed_test.csv")
test_dataset = Dataset.from_pandas(test_df)

def tokenize_tail(examples):
    outputs = tokenizer(examples["text"], add_special_tokens=False)
    input_ids = []
    for ids in outputs["input_ids"]:
        processed = ids[-510:] if len(ids) > 510 else ids
        input_ids.append([tokenizer.bos_token_id] + processed + [tokenizer.eos_token_id])
    return {"input_ids": input_ids}

test_dataset = test_dataset.map(tokenize_tail, batched=True)

tester = Trainer(model=model, data_collator=DataCollatorWithPadding(tokenizer=tokenizer))
raw_preds = tester.predict(test_dataset)

y_pred = np.argmax(raw_preds.predictions, axis=1)
y_true = test_df['label'].values

print("--- Test Set Results (Downsampled Model) ---")
print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (563 > 512). Running this sequence through the model will result in indexing errors


--- Test Set Results (Downsampled Model) ---
              precision    recall  f1-score   support

    negative       0.50      0.90      0.64        10
     neutral       0.00      0.00      0.00        10
    positive       0.67      0.80      0.73        10

    accuracy                           0.57        30
   macro avg       0.39      0.57      0.46        30
weighted avg       0.39      0.57      0.46        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
